In [5]:
import os
import csv
import numpy as np                  # For doing math
import matplotlib.pyplot as plt     # For plotting
import matplotlib.dates as mdates   # For formatting dates when plotting
import matplotlib.colors as colors  # For truncating colorbars
import matplotlib.style as style
import xarray as xr                 # For dealing with netCDF data
import pandas as pd                 # A quick way to deal with time stamps
import netCDF4 as nc                # Another way to deal with netCDF data
import glob
import metpy.calc as mpcalc
import datetime
import matplotlib.units as munits
import scipy.interpolate
import scipy.signal as sig
import scipy.stats as stats
import sys
import scipy.io as sio

print('Should clean up redundancies')
from metpy.cbook import get_test_data
from metpy.plots import Hodograph, SkewT
from metpy.units import units
from scipy.signal import savgol_filter
from scipy.stats import binned_statistic
from matplotlib.gridspec import GridSpec
# from numpy.random import seed
# from numpy.random import rand

print('Done importing modules now')

import warnings
warnings.filterwarnings('ignore')

import matplotlib
from netCDF4 import Dataset
from xarray import DataArray
from wrf import (getvar, interplevel, vertcross, 
                 vinterp, ALL_TIMES, extract_global_attrs)
import cartopy.crs as ccrs          # For plotting maps
import cartopy.feature as cfeature  # For plotting maps

Should clean up redundancies
Done importing modules now


In [6]:
# Streamlined destagger

def destagger(var, stagger_dim):
    '''
    Modified From wrf-python https://github.com/NCAR/wrf-python/blob/b40d1d6e2d4aea3dd2dda03aae18e268b1e9291e/src/wrf/destag.py 
    '''
    var_shape = var.shape
    num_dims = var.ndim
    stagger_dim_size = var_shape[stagger_dim]

    full_slice = slice(None)
    slice1 = slice(0, stagger_dim_size - 1, 1)
    slice2 = slice(1, stagger_dim_size, 1)

    dim_ranges_1 = [full_slice] * num_dims
    dim_ranges_2 = [full_slice] * num_dims

    dim_ranges_1[stagger_dim] = slice1
    dim_ranges_2[stagger_dim] = slice2

    result = .5*(var[tuple(dim_ranges_1)] + var[tuple(dim_ranges_2)])

    return result

In [7]:
def choose_domain_d02_only(run_name):
    name = run_name.lower()
    if "hrrr" in name:
        return "d01"
    elif "era5" in name:
        return "d02"
    elif "gfs" in name:
        return "d02"
    else:
        raise ValueError(f"Unknown run type in d02_only: {run_name}")

def choose_domain_all_runs(run_name):
    name = run_name.lower()

    # enforce: all_runs should only be 3dpbl
    if "3dpbl" not in name:
        raise ValueError(f"Non-3dpbl run passed to choose_domain_all_runs: {run_name}")

    # existing logic: HRRR → d02, everything else → d03
    if "hrrr" in name:
        return "d02"
    else:
        return "d03"

In [11]:
import os
import numpy as np
import xarray as xr
from datetime import datetime, timedelta
import traceback

# ========================== OUTPUT ROOT DIR ==========================

ROOT_OUT = "/glade/work/mhafsah/Analysis_Proj_1"
os.makedirs(ROOT_OUT, exist_ok=True)

DAY_STR   = "2020-05-15"   # LLJ day
t0_hour   = 14
t1_hour   = 23
dt_minutes = 10

height_levels = np.arange(20, 201, 20)


# ========================== HELPERS FOR EXPECTED FILES ==========================

def _make_expected_ts(day_str, t0=14, t1=23, dt_minutes=10):
    """
    Build timestamps from t0 to t1 inclusive, every dt_minutes.
    For 14–23 UTC with 10-min steps, this gives 55 entries.
    """
    day = datetime.strptime(day_str, "%Y-%m-%d")
    start = day.replace(hour=t0, minute=0, second=0)
    end   = day.replace(hour=t1, minute=0, second=0)

    out = []
    cur = start
    while cur <= end:
        out.append(cur.strftime("%Y-%m-%d_%H:%M:%S"))
        cur += timedelta(minutes=dt_minutes)
    return out


def _candidate_path(run_dir, dom, ts):
    """
    Expected file name:
      custom_wrfout_d0X_YYYY-MM-DD_HH:MM:SS
    """
    fname = f"custom_wrfout_{dom}_{ts}"
    return os.path.join(run_dir, fname)

In [13]:
# ========================== WHICH SET ARE WE RUNNING? ==========================

USE_ALL_RUNS = False   # set False if you want d02_only

if USE_ALL_RUNS:
    base_src   = "/glade/derecho/scratch/mhafsah/sensitivity_trial2/all_runs"
    choose_domain = choose_domain_all_runs
    ws_base    = os.path.join(ROOT_OUT, "WS_all_runs")
    wd_base    = os.path.join(ROOT_OUT, "WD_all_runs")
else:
    base_src   = "/glade/derecho/scratch/mhafsah/sensitivity_trial2/d02_only"
    choose_domain = choose_domain_d02_only
    ws_base    = os.path.join(ROOT_OUT, "WS_d02")
    wd_base    = os.path.join(ROOT_OUT, "WD_d02")

os.makedirs(ws_base, exist_ok=True)
os.makedirs(wd_base, exist_ok=True)

# ========================== MAIN LOOP OVER RUN DIRECTORIES ==========================

for run in sorted(os.listdir(base_src)):
    run_dir = os.path.join(base_src, run)
    if not os.path.isdir(run_dir):
        continue

    try:
        dom = choose_domain(run)  # will raise ValueError for non-3dpbl in all_runs
    except ValueError as e:
        print("Skipping", run, ":", e)
        continue

    print(f"\n=== {run} → using {dom} ===")

    # 55 timestamps we expect
    expected_ts = _make_expected_ts(DAY_STR, t0=t0_hour, t1=t1_hour, dt_minutes=dt_minutes)
    if len(expected_ts) != 55:
        print(f"  ERROR: expected_ts len is {len(expected_ts)} (should be 55). Check time logic.")
        continue

    # ---------- find a first existing file to get grid indices (E05/E06) ----------

    first_file = None
    for ts in expected_ts:
        p = _candidate_path(run_dir, dom, ts)
        if os.path.exists(p):
            first_file = p
            break

    if first_file is None:
        print("  ERROR: none of the expected files exist (likely wrong dom or naming)")
        continue

    try:
        ds0 = xr.open_dataset(first_file, engine="netcdf4")

        # Buoy locations
        e05_lon, e05_lat = -72.72, 39.97
        e06_lon, e06_lat = -73.43, 39.55

        # XLAT / XLONG may have Time dimension; use first (and only) time slice
        lats = ds0["XLAT"]
        if "Time" in lats.dims:
            lats = lats.isel(Time=0)

        longs = ds0["XLONG"]
        if "Time" in longs.dims:
            longs = longs.isel(Time=0)

        km_lon = 85
        km_lat = 111

        e05_abslatdiff = np.abs(lats - e05_lat)
        e05_abslondiff = np.abs(longs - e05_lon)
        e05_abslatdiff_km = e05_abslatdiff * km_lat
        e05_abslondiff_km = e05_abslondiff * km_lon
        e05_distdiff_km = (e05_abslatdiff_km**2 + e05_abslondiff_km**2) ** 0.5
        e05_s_n, e05_w_e = np.where(e05_distdiff_km == np.min(e05_distdiff_km))

        e06_abslatdiff = np.abs(lats - e06_lat)
        e06_abslondiff = np.abs(longs - e06_lon)
        e06_abslatdiff_km = e06_abslatdiff * km_lat
        e06_abslondiff_km = e06_abslondiff * km_lon
        e06_distdiff_km = (e06_abslatdiff_km**2 + e06_abslondiff_km**2) ** 0.5
        e06_s_n, e06_w_e = np.where(e06_distdiff_km == np.min(e06_distdiff_km))

        print("  s_n index of buoy E05 =", e05_s_n)
        print("  w_e index of buoy E05 =", e05_w_e)
        print("  s_n index of buoy E06 =", e06_s_n)
        print("  w_e index of buoy E06 =", e06_w_e)

    except Exception as e:
        print(f"  ERROR: can't read grid/indices from {os.path.basename(first_file)}: {e}")
        traceback.print_exc()
        continue
    finally:
        try:
            ds0.close()
        except Exception:
            pass

    # ---------- allocate WS/WD arrays (55 time steps × 10 heights) ----------

    WS_E05_WRF = np.full((55, 10), np.nan, dtype=float)
    WS_E06_WRF = np.full((55, 10), np.nan, dtype=float)
    WD_E05_WRF = np.full((55, 10), np.nan, dtype=float)
    WD_E06_WRF = np.full((55, 10), np.nan, dtype=float)

    missing_ts = []
    failed_ts  = []

    # ========================== LOOP OVER 55 EXPECTED FILES ==========================

    for k, ts in enumerate(expected_ts):
        fpath = _candidate_path(run_dir, dom, ts)
        if not os.path.exists(fpath):
            missing_ts.append(ts)
            continue

        try:
            ds = xr.open_dataset(fpath, engine="netcdf4")

            # --- WS processing for this one file (time index k) ---

            height = ((ds["PH"][:] + ds["PHB"][:]) / 9.81 - ds["HGT"][:])
            u = ds["U"]
            v = ds["V"]

            # Destaggering process
            u_destag = destagger(u, 3)
            u_destag = u_destag.rename({"west_east_stag": "west_east"})

            v_destag = destagger(v, 2)
            v_destag = v_destag.rename({"south_north_stag": "south_north"})

            height_destag = destagger(height, 1)
            height_destag = height_destag.rename({"bottom_top_stag": "bottom_top"})

            # overwrite staggered
            u = u_destag
            v = v_destag
            height = height_destag

            # ------------- WS -------------
            ws = (u**2 + v**2)**0.5

            interp_ws = {}
            for level in height_levels:
                interp_ws_level = interplevel(ws, height, level)
                if "level" in interp_ws_level.coords:
                    interp_ws_level = interp_ws_level.drop_vars("level")
                interp_ws[f"ws_{level}m"] = interp_ws_level

            interp_ws_dataset = xr.Dataset(interp_ws)

            interp_ws_dataset_E05 = interp_ws_dataset.isel(
                south_north=e05_s_n[0],
                west_east=e05_w_e[0]
            )
            interp_ws_dataset_E06 = interp_ws_dataset.isel(
                south_north=e06_s_n[0],
                west_east=e06_w_e[0]
            )

            # Fill WS row k
            WS_E05_WRF[k, 0] = np.asarray(interp_ws_dataset_E05["ws_20m"].values).squeeze()
            WS_E06_WRF[k, 0] = np.asarray(interp_ws_dataset_E06["ws_20m"].values).squeeze()
            WS_E05_WRF[k, 1] = np.asarray(interp_ws_dataset_E05["ws_40m"].values).squeeze()
            WS_E06_WRF[k, 1] = np.asarray(interp_ws_dataset_E06["ws_40m"].values).squeeze()
            WS_E05_WRF[k, 2] = np.asarray(interp_ws_dataset_E05["ws_60m"].values).squeeze()
            WS_E06_WRF[k, 2] = np.asarray(interp_ws_dataset_E06["ws_60m"].values).squeeze()
            WS_E05_WRF[k, 3] = np.asarray(interp_ws_dataset_E05["ws_80m"].values).squeeze()
            WS_E06_WRF[k, 3] = np.asarray(interp_ws_dataset_E06["ws_80m"].values).squeeze()
            WS_E05_WRF[k, 4] = np.asarray(interp_ws_dataset_E05["ws_100m"].values).squeeze()
            WS_E06_WRF[k, 4] = np.asarray(interp_ws_dataset_E06["ws_100m"].values).squeeze()
            WS_E05_WRF[k, 5] = np.asarray(interp_ws_dataset_E05["ws_120m"].values).squeeze()
            WS_E06_WRF[k, 5] = np.asarray(interp_ws_dataset_E06["ws_120m"].values).squeeze()
            WS_E05_WRF[k, 6] = np.asarray(interp_ws_dataset_E05["ws_140m"].values).squeeze()
            WS_E06_WRF[k, 6] = np.asarray(interp_ws_dataset_E06["ws_140m"].values).squeeze()
            WS_E05_WRF[k, 7] = np.asarray(interp_ws_dataset_E05["ws_160m"].values).squeeze()
            WS_E06_WRF[k, 7] = np.asarray(interp_ws_dataset_E06["ws_160m"].values).squeeze()
            WS_E05_WRF[k, 8] = np.asarray(interp_ws_dataset_E05["ws_180m"].values).squeeze()
            WS_E06_WRF[k, 8] = np.asarray(interp_ws_dataset_E06["ws_180m"].values).squeeze()
            WS_E05_WRF[k, 9] = np.asarray(interp_ws_dataset_E05["ws_200m"].values).squeeze()
            WS_E06_WRF[k, 9] = np.asarray(interp_ws_dataset_E06["ws_200m"].values).squeeze()

            # ------------- WD -------------
            wd = 180.0 + (180.0 / np.pi) * np.arctan2(v, u)

            interp_wd = {}
            for level in height_levels:
                interp_wd_level = interplevel(wd, height, level)
                if "level" in interp_wd_level.coords:
                    interp_wd_level = interp_wd_level.drop_vars("level")
                interp_wd[f"wd_{level}m"] = interp_wd_level

            interp_wd_dataset = xr.Dataset(interp_wd)

            interp_wd_dataset_E05 = interp_wd_dataset.isel(
                south_north=e05_s_n[0],
                west_east=e05_w_e[0]
            )
            interp_wd_dataset_E06 = interp_wd_dataset.isel(
                south_north=e06_s_n[0],
                west_east=e06_w_e[0]
            )

            # Fill WD row k
            WD_E05_WRF[k, 0] = np.asarray(interp_wd_dataset_E05["wd_20m"].values).squeeze()
            WD_E06_WRF[k, 0] = np.asarray(interp_wd_dataset_E06["wd_20m"].values).squeeze()
            WD_E05_WRF[k, 1] = np.asarray(interp_wd_dataset_E05["wd_40m"].values).squeeze()
            WD_E06_WRF[k, 1] = np.asarray(interp_wd_dataset_E06["wd_40m"].values).squeeze()
            WD_E05_WRF[k, 2] = np.asarray(interp_wd_dataset_E05["wd_60m"].values).squeeze()
            WD_E06_WRF[k, 2] = np.asarray(interp_wd_dataset_E06["wd_60m"].values).squeeze()
            WD_E05_WRF[k, 3] = np.asarray(interp_wd_dataset_E05["wd_80m"].values).squeeze()
            WD_E06_WRF[k, 3] = np.asarray(interp_wd_dataset_E06["wd_80m"].values).squeeze()
            WD_E05_WRF[k, 4] = np.asarray(interp_wd_dataset_E05["wd_100m"].values).squeeze()
            WD_E06_WRF[k, 4] = np.asarray(interp_wd_dataset_E06["wd_100m"].values).squeeze()
            WD_E05_WRF[k, 5] = np.asarray(interp_wd_dataset_E05["wd_120m"].values).squeeze()
            WD_E06_WRF[k, 5] = np.asarray(interp_wd_dataset_E06["wd_120m"].values).squeeze()
            WD_E05_WRF[k, 6] = np.asarray(interp_wd_dataset_E05["wd_140m"].values).squeeze()
            WD_E06_WRF[k, 6] = np.asarray(interp_wd_dataset_E06["wd_140m"].values).squeeze()
            WD_E05_WRF[k, 7] = np.asarray(interp_wd_dataset_E05["wd_160m"].values).squeeze()
            WD_E06_WRF[k, 7] = np.asarray(interp_wd_dataset_E06["wd_160m"].values).squeeze()
            WD_E05_WRF[k, 8] = np.asarray(interp_wd_dataset_E05["wd_180m"].values).squeeze()
            WD_E06_WRF[k, 8] = np.asarray(interp_wd_dataset_E06["wd_180m"].values).squeeze()
            WD_E05_WRF[k, 9] = np.asarray(interp_wd_dataset_E05["wd_200m"].values).squeeze()
            WD_E06_WRF[k, 9] = np.asarray(interp_wd_dataset_E06["wd_200m"].values).squeeze()

        except Exception as e:
            failed_ts.append((ts, os.path.basename(fpath), str(e)))
            print(f"  WARN: failed at {ts} ({os.path.basename(fpath)}): {e}")
        finally:
            try:
                ds.close()
            except Exception:
                pass

    # ========================== SAVE PER-RUN OUTPUT ==========================

    ws_out_dir = os.path.join(ws_base, run)
    wd_out_dir = os.path.join(wd_base, run)
    os.makedirs(ws_out_dir, exist_ok=True)
    os.makedirs(wd_out_dir, exist_ok=True)

    np.save(os.path.join(ws_out_dir, "WS_E05.npy"), WS_E05_WRF)
    np.save(os.path.join(ws_out_dir, "WS_E06.npy"), WS_E06_WRF)
    np.save(os.path.join(wd_out_dir, "WD_E05.npy"), WD_E05_WRF)
    np.save(os.path.join(wd_out_dir, "WD_E06.npy"), WD_E06_WRF)

    print(f"  saved: {ws_out_dir}/WS_E05.npy  shape={WS_E05_WRF.shape}")
    print(f"  saved: {ws_out_dir}/WS_E06.npy  shape={WS_E06_WRF.shape}")
    print(f"  saved: {wd_out_dir}/WD_E05.npy  shape={WD_E05_WRF.shape}")
    print(f"  saved: {wd_out_dir}/WD_E06.npy  shape={WD_E06_WRF.shape}")

    if missing_ts:
        print(f"  MISSING files ({len(missing_ts)}/55):")
        for ts in missing_ts[:10]:
            print("   ", ts)
        if len(missing_ts) > 10:
            print("   ...")

    if failed_ts:
        print(f"  FAILED processing ({len(failed_ts)}):")
        for ts, fn, msg in failed_ts[:5]:
            print(f"   {ts}  {fn}  {msg}")
        if len(failed_ts) > 5:
            print("   ...")


=== era5_myj_vert54_d02 → using d02 ===
  s_n index of buoy E05 = [205]
  w_e index of buoy E05 = [362]
  s_n index of buoy E06 = [174]
  w_e index of buoy E06 = [321]
  saved: /glade/work/mhafsah/Analysis_Proj_1/WS_d02/era5_myj_vert54_d02/WS_E05.npy  shape=(55, 10)
  saved: /glade/work/mhafsah/Analysis_Proj_1/WS_d02/era5_myj_vert54_d02/WS_E06.npy  shape=(55, 10)
  saved: /glade/work/mhafsah/Analysis_Proj_1/WD_d02/era5_myj_vert54_d02/WD_E05.npy  shape=(55, 10)
  saved: /glade/work/mhafsah/Analysis_Proj_1/WD_d02/era5_myj_vert54_d02/WD_E06.npy  shape=(55, 10)

=== era5_myj_vert80_d02 → using d02 ===
  s_n index of buoy E05 = [205]
  w_e index of buoy E05 = [362]
  s_n index of buoy E06 = [174]
  w_e index of buoy E06 = [321]
  saved: /glade/work/mhafsah/Analysis_Proj_1/WS_d02/era5_myj_vert80_d02/WS_E05.npy  shape=(55, 10)
  saved: /glade/work/mhafsah/Analysis_Proj_1/WS_d02/era5_myj_vert80_d02/WS_E06.npy  shape=(55, 10)
  saved: /glade/work/mhafsah/Analysis_Proj_1/WD_d02/era5_myj_vert80_